# Why Should This Work?

**This notebook is the middle of today's class.** The slides set up the question; you come
here for about 30 minutes, in pairs; then we go back to the slides.

- Work through **§1 to §4 in order**. Cells marked 🔒 run as they are.
- You write **one prompt** (§3) and **one line** (§4).
- Stop at **⏸ Back to the slides**. Everything after it is for after class.

## 📋 Today's Plan

**In class**
1. [§1 · The gap: check your first prediction](#gap)
2. [§2 · Count ideas, not papers: check your second prediction](#count)
3. [§3 · 🎯 Prompt it: what was the worst drawdown?](#prompt)
4. [§4 · 🛠️ Your strategy: its neighbours and its worst episode](#ho1)
5. [⏸ Back to the slides](#back)

**After class**

6. [§5 · Self-study: the economics from the slides](#selfstudy)
7. [🎯 Challenge: another family](#challenge) — *homework*
8. [Key takeaways](#takeaways)

---

## 🛠️ Setup

In [1]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

# The 29 long-short strategies from the backtesting lecture, plus the published t-statistics.
L    = pd.read_parquet(f"{BASE}/longshort_29.parquet")
menu = pd.read_csv(f"{BASE}/signal_menu.csv").set_index('Acronym')
ff   = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)
tstat  = lambda x: x.mean() / x.std() * np.sqrt(len(x))

print(f"{L.shape[1]} strategies, {len(L)} months, "
      f"{L.index[0]:%Y-%m} to {L.index[-1]:%Y-%m}")

29 strategies, 251 months, 1980-02 to 2000-12


---

## §1 · The gap: check your first prediction <a id="gap"></a>

Every one of these strategies was published in a peer-reviewed journal with a significant
t-statistic. On the slides you predicted how many of the 27 that report one clear the
**Bonferroni bar** in our sample, 1980 to 2000. Type your answer, then run the 🔒 cell.

In [2]:
my_guess = None      # ← your prediction from the slide: how many of the 27 clear the bar?

In [3]:
#@title 🔒 What the journals reported, and what we get
rows = [(s, menu.loc[s, 'T-Stat'], tstat(L[s].dropna()))
        for s in L.columns if s in menu.index and pd.notna(menu.loc[s, 'T-Stat'])]
d = pd.DataFrame(rows, columns=['signal', 'published_t', 'our_t']).set_index('signal')

bar = abs(norm.ppf(0.025 / len(d)))          # Bonferroni bar, from the backtesting lecture
print(f"{len(d)} strategies with a published t-statistic\n")
print(f"  mean published t          {d.published_t.mean():6.2f}")
print(f"  mean t in our sample      {d.our_t.mean():6.2f}")
print(f"  clear |t| > 1.96          {(d.our_t > 1.96).sum():3d} of {len(d)}")
print(f"  clear the Bonferroni bar  {(d.our_t > bar).sum():3d} of {len(d)}   (bar = {bar:.2f})"
      + (f"   <- you guessed {my_guess}" if my_guess is not None else ""))
print(f"  come out the WRONG SIGN   {(d.our_t < 0).sum():3d} of {len(d)}")

r = sm.OLS(d.our_t, sm.add_constant(d.published_t)).fit()
print(f"\n  our_t = {r.params.iloc[0]:.2f} + {r.params.iloc[1]:.2f} x published_t"
      f"   R-squared {r.rsquared:.2f}")

27 strategies with a published t-statistic

  mean published t            5.82
  mean t in our sample        1.87
  clear |t| > 1.96           15 of 27
  clear the Bonferroni bar    6 of 27   (bar = 3.11)
  come out the WRONG SIGN     6 of 27

  our_t = 0.57 + 0.22 x published_t   R-squared 0.12


**What to look at**

- How many clear the bar, and how many come out with the **wrong sign**.
- The last line: how much the published t-statistic tells you about ours. That is the R-squared.

Why this happens is the first thing we do back on the slides.

---

## §2 · Count ideas, not papers: check your second prediction <a id="count"></a>

Three volatility papers, two sets of authors. On the slides you guessed how correlated
their long-short returns are.

In [4]:
my_corr_guess = None      # ← your answer from the slide: "A", "B", "C" or "D"

In [5]:
#@title 🔒 Three papers, two sets of authors, three names
fam = ['IdioVol3F', 'RealizedVol', 'MaxRet']
info = menu.loc[fam, ['Authors', 'Year', 'T-Stat']]
info['our_t'] = [tstat(L[s].dropna()) for s in fam]
print(info.to_string(), "\n")
print("correlation of the three long-short return series:")
print(L[fam].corr().round(2).to_string())

c = L.corr().values
pairs = c[np.triu_indices(c.shape[0], 1)]
print(f"\nacross all 29 strategies, the average correlation in absolute value is {np.abs(pairs).mean():.2f}"
      + (f"\nyou guessed {my_corr_guess}" if my_corr_guess is not None else ""))

                                Authors  Year  T-Stat     our_t
Acronym                                                        
IdioVol3F                    Ang et al.  2006    3.10  2.607801
RealizedVol                  Ang et al.  2006    2.86  2.160616
MaxRet       Bali, Cakici, and Whitelaw  2011    2.83  1.724120 

correlation of the three long-short return series:
             IdioVol3F  RealizedVol  MaxRet
IdioVol3F         1.00         0.96    0.94
RealizedVol       0.96         1.00    0.97
MaxRet            0.94         0.97    1.00

across all 29 strategies, the average correlation in absolute value is 0.20


**What to look at**

- Inside the family, 0.94 to 0.97. These are not three strategies: they are one strategy,
  measured three ways. Holding all three triples one position.
- Across all 29, the typical correlation is about 0.20 in absolute value, so mostly they are
  different bets. Inside a family they are not.

> **📌 Count ideas, not papers.** Before any multiple-testing correction, ask how many distinct
> economic mechanisms you tried. That number is smaller than the number of regressions you ran.

---

## §3 · 🎯 Prompt it: what was the worst drawdown? <a id="prompt"></a>

Take the first of the three volatility strategies, `IdioVol3F`: long the quiet stocks, short
the volatile ones. On the slides afterwards we look at what holding it felt like. First, one
number.

> **🤔 The question.** *"What was this strategy's worst drawdown?"*
>
> Write the prompt. Before you do, think about what this portfolio **is** (a long-short) and
> what you are dividing by.

In [6]:
# === YOUR TURN ===
x = L['IdioVol3F'].dropna()          # the strategy's monthly returns

MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----

Now run the check. It computes the same question four ways, and all four are correct code.

In [7]:
#@title 🔒 Check — four defensible answers
def maxdd(s):
    c = (1 + s).cumprod(); dd = c / c.cummax() - 1
    return dd.min(), dd.idxmin()

cs = x.cumsum()
rows = [("compounded, funded 1-for-1 with equity",) + maxdd(x),
        ("cumulative sum of returns (no capital base)",
         (cs - cs.cummax()).min(), (cs - cs.cummax()).idxmin()),
        ("compounded at 2x leverage",) + maxdd(2 * x),
        ("compounded at 3x leverage",) + maxdd(3 * x)]

print("«What was this strategy's worst drawdown?»\n")
for lab, v, t in rows:
    print(f"  {lab:46s}{v:8.1%}   trough {t:%Y-%m}")
spread = max(v for _, v, _ in rows) - min(v for _, v, _ in rows)
print(f"\n  four correct answers, {abs(spread) * 100:.0f} percentage points apart")

«What was this strategy's worst drawdown?»

  compounded, funded 1-for-1 with equity          -74.9%   trough 2000-02
  cumulative sum of returns (no capital base)    -120.9%   trough 2000-02
  compounded at 2x leverage                       -96.4%   trough 2000-02
  compounded at 3x leverage                      -100.0%   trough 2000-06

  four correct answers, 46 percentage points apart


### Read the code

The function `maxdd` is three lines, and each one carries a decision:

- `(1 + s).cumprod()` is the value of \$1 invested at the start, month by month. **This is where
  "funded 1-for-1 with equity" sneaks in**: it assumes you put up \$1 of capital for every \$1
  long and \$1 short.
- `c.cummax()` is the highest value reached so far.
- `c / c.cummax() - 1` is how far below its previous peak the strategy sits each month. The
  minimum is the worst drawdown.

The second row uses `cs - cs.cummax()` on a running **sum**: no compounding and no capital
base, which is why it can fall below −100%.

> **📌 A long-short has no capital of its own, so a drawdown is undefined until you say what
> capital you posted.** A terse prompt returns `(1 + r).cumprod()` and never says so.
> We pick up the 2× and 3× rows on the slides.

---

## §4 · 🛠️ Your strategy: its neighbours and its worst episode <a id="ho1"></a>

Two things about your group's signal, one measured and one argued.

> **🤔 Predict first.** How correlated do you think your signal is with the closest other
> strategy in the 29? Write the number down before you run it.

In [8]:
# === EDIT + YOUR TURN ===
MY_SIGNAL = "GP"      # ← your group's signal

# 1. Who are your neighbours? The three most correlated strategies.
nb = L.corr()[MY_SIGNAL].drop(MY_SIGNAL).sort_values(ascending=False).head(3)
print(f"{MY_SIGNAL} — closest neighbours\n")
for s, c in nb.items():
    print(f"  {s:22s} corr {c:+.2f}   {menu.loc[s,'Cat.Economic'] if s in menu.index else ''}")

# 2. Your worst episode. State the definition — don't let the code choose for you.
mine = L[MY_SIGNAL].dropna()
dd_value, dd_date = ____      # hint: maxdd(mine), compounded and funded 1-for-1
print(f"\n  worst drawdown {dd_value:.1%}, trough {dd_date:%Y-%m}")
print(f"  market that month: {ff.loc[dd_date,'Mkt-RF']:+.1%}")

GP — closest neighbours

  roaq                   corr +0.46   profitability
  CBOperProf             corr +0.44   profitability
  OrgCap                 corr +0.33   R&D


NameError: name '____' is not defined

### Compare with the room

- **How close is your nearest neighbour?** Above 0.8 and you should be able to say what the
  shared mechanism is. If you cannot, you do not yet know what your signal measures.
- **When was your worst episode?** Compare trough dates across the room. If half the class
  troughs in the same month, the class does not hold ten strategies.
- **Now the hard one, for the slides:** who is on the other side of your trade, and what makes
  them willing? A constraint, a preference, or an error?

---

## ⏸ Back to the slides <a id="back"></a>

That is the end of the in-class part. Everything below is for after class: the economics from
the slides written out, the homework, and the takeaways.

---

## §5 · Self-study: the economics from the slides <a id="selfstudy"></a>

### Three explanations for the gap

| | Mechanism | So the effect… |
|---|---|---|
| **Selection** | journals print what worked; the failures stay in a drawer | never existed |
| **Overfitting** | the authors searched too | never existed |
| **Decay** | it was real; publication revealed it; investors traded it away | existed, and is gone |

All three produce the same replication table, so the data cannot tell you which. Two say
never trade it; one says you are late.

### Risk for whom?

Risk is not a property of an asset. It is a property of an asset **for a particular
investor**. You are paid to hold something that pays off badly in bad times, when you can
least afford it. The CAPM is one narrow version of this: it declares that bad times means the
market falling. But bad times are about people: a lost job, a house that falls with the
market, a credit line that disappears.

If you work in the auto industry, your income already moves with autos, so hold the market
**minus** autos.

| | | What you should do |
|---|---|---|
| **You are the average investor** | your bad times are the market's | hold the market |
| **You genuinely differ** | what scares them does not scare you | take the other side, and get paid |
| **You think you differ, and do not** | the most common case | you find out at the worst time |

**As you scale, you become the investor bearing the risk.** "Risky for them, not for me" is a
statement about a small position. And if everyone held it, prices would adjust and the premium
would shrink: can the world support everyone picking up this \$100 bill?

### A high expected return is a low price

"These stocks earn more" is the same sentence as "these stocks are cheap". Someone is selling
to you at that price.

| | Why the price is low | What you are buying |
|---|---|---|
| **Cash flows** | the market expects the business to do badly, and is right | a bad business at a fair price |
| **Risk** | it pays off badly when investors care most | compensation for bearing something real |

If you cannot name who is selling to you, and why, you are the one being selected.

### Low volatility, all the way down

**The counterparty.** Many investors want equity-like returns and cannot borrow to get them:
mandates forbid leverage, regulation penalises it, margin is expensive. To reach for return
they buy risky stocks, which bids up the volatile names and leaves the quiet ones cheap. Their
reason is a **constraint**, not a mistake, and constraints do not go away because someone
points them out. So the strategy is risky for you only if you are unconstrained: if you can
borrow, and if you can hold.

In [ ]:
#@title 🔒 What holding low volatility actually felt like
cagr = lambda s: (1 + s).prod() ** (12 / len(s)) - 1

print(f"{'':14s}{'compounded':>12s}{'vol':>9s}")
for lab, sl in [('1980-1997', x.loc[:'1997-12-31']), ('1998', x.loc['1998']),
                ('1999', x.loc['1999']), ('2000', x.loc['2000'])]:
    ann = cagr(sl) if lab == '1980-1997' else (1 + sl).prod() - 1
    print(f"  {lab:12s}{ann:>11.1%}{sl.std()*np.sqrt(12):>9.1%}")

w = (1 + x).cumprod()
peak = w.idxmax()
trough = w.loc[peak:].idxmin()
print(f"\n  $1 in {x.index[0]:%Y-%m}  ->  ${w.loc[peak]:.2f} at the peak ({peak:%Y-%m})")
print(f"  {'':13s}      ->  ${w.loc[trough]:.2f} at the trough ({trough:%Y-%m})")
print(f"  {'':13s}      ->  ${w.iloc[-1]:.2f} at the end ({x.index[-1]:%Y-%m})")

fig, ax = plt.subplots()
for k, colour in [(1, '#333333'), (2, '#2E5C8A'), (3, '#B03A2E')]:
    path = (1 + k * x.loc['1997':]).cumprod().clip(lower=0)
    ax.plot(path.index, path.values, lw=1.4, color=colour, label=f'{k}x leverage')
ax.set_ylabel('growth of $1 invested in Jan 1997'); ax.legend(loc='upper left')
ax.set_title('The same returns at 1x, 2x and 3x leverage', loc='left', fontsize=11)
plt.tight_layout(); plt.show()

print(f"  Sharpe 1980-1990 (the backtesting lecture's estimate sample)  {sharpe(x.loc[:'1990-12-31']):.2f}")
print(f"  Sharpe 1991-2000                                             {sharpe(x.loc['1991-01-31':]):.2f}")

Eighteen years of about 15% a year, then 1999 takes 60% in twelve months. The strategy is short
lottery stocks, and in a bubble lottery stocks are exactly what runs. At 2× leverage you lose
96%; at 3× you are wiped out in June 2000, before the recovery.

This was the backtesting lecture's in-sample winner, and we filed its collapse under
overfitting. It may instead be the premium being paid for: the bad state the story predicts,
arriving. **Nothing in the returns tells you which. Only a story written down beforehand does.**

### The worked problem from the slides

Four months of a long-short: +10%, −20%, −20%, +5%. Work it by hand first, then check here.

In [ ]:
#@title 🔒 Check the worked problem
path = pd.Series([0.10, -0.20, -0.20, 0.05])
s = path.cumsum()
print(f"  (a) 1x, compounded   {maxdd(path)[0]:.0%}")
print(f"  (b) 2x, compounded   {maxdd(2 * path)[0]:.0%}")
print(f"  (c) summing returns  {(s - s.cummax()).min() * 100:.0f} points")

### Five questions before you trade a signal

1. **Who is on the other side**, and what makes them willing?
2. Is their reason a **constraint, a preference, or an error**? Constraints persist; errors get arbitraged.
3. Is the risk one **you** can bear, at the size you intend to trade?
4. What would you have to see to stop believing? **Name it now**, while it is cheap.
5. If the answer to (1) is "I don't know", **you are the counterparty**.

---

---

## 🎯 Challenge: Another Family <a id="challenge"></a>

*Homework — due before the next class.*

§2 took the volatility family apart. Do the same to **valuation**: `BM`
(Stattman 1980), `EP` (Basu 1977) and `EntMult` (Loughran and Wellman 2011).
Three papers, three decades, one economic idea — or so it is usually described.

Throughout, the **composite** means the equal-weighted average of the three
monthly return series.

### Q1 — How many ideas is this?

Report the **mean pairwise correlation** of the three long-short return series —
the average of the three off-diagonal entries of the correlation matrix.

> **📌 Required variable names:**
> ```python
> fam_corr = ____   # mean pairwise correlation of BM, EP, EntMult
> ```

In [ ]:
# Your work here


fam_corr = ____

print(f"mean pairwise correlation: {fam_corr:.2f}")

### Q2 — The worst episode

Maximum drawdown of the composite, **compounded, funded 1-for-1 with equity** —
the first row of the table in class. Report it as a negative decimal
(−0.30 means −30%).

Then look at *when* the trough falls, and compare it with the low-volatility
trough from class.

> **📌 Required variable names:**
> ```python
> val_dd = ____     # maximum drawdown, negative decimal
> ```

In [ ]:
# Your work here


val_dd = ____

print(f"max drawdown: {val_dd:.1%}")

### Q3 — Is it a hedge?

Regress the composite on the excess market return and report the **beta**.

Then ask what it means. A negative beta says the strategy makes money when the
market falls. If that is true, the CAPM says it should earn *less* than the
risk-free rate, not more — so a positive average return alongside a negative beta
is a puzzle the model cannot absorb.

> **📌 Required variable names:**
> ```python
> val_beta = ____   # market beta of the composite
> ```

In [ ]:
# Your work here


val_beta = ____

print(f"market beta: {val_beta:+.2f}")

### Q4 — The memo

> **📝 Your task — maximum eight sentences.**
>
> Your PM read a note claiming the firm holds "three independent value signals"
> and wants to know whether that is true and whether to add a fourth.
>
> Say how many bets those three actually are, and how you know. Say when the
> composite had its worst episode, and what it means that the low-volatility
> strategy — a completely different economic idea — had its worst episode in the
> same month. Then answer the question the whole lecture has been building
> toward: **who is on the other side of the value trade, and is their reason a
> constraint, a preference, or an error?** Say which, and say what you would have
> to observe to change your mind.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["fam_corr", "val_dd", "val_beta", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L9_WhyShouldThisWork_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **A replication gap has three explanations** (selection, overfitting, decay), and the data
   cannot separate them.
2. **Count ideas, not papers.** Three volatility papers correlate 0.94 to 0.97: one trade with
   three citations.
3. **Risk is a property of an asset for an investor.** Deviate from the market only if you
   differ from the average investor.
4. **A high expected return is a low price.** Ask who is selling, and why. A constraint
   persists; an error gets arbitraged.
5. **As you scale, you become the investor bearing the risk.**
6. **A drawdown is undefined for a long-short until you say what capital you posted.** Four
   correct answers, 46 points apart, from the same returns.
7. **Nothing in the returns tells you whether a collapse is overfitting or the premium being
   paid.** Only a story written down beforehand does.

---

## 📎 Appendix <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — the full replication table
# ═══════════════════════════════════════════════════════════════════════
out = d.copy()
out['ratio']  = out.published_t / out.our_t
out['sharpe'] = [sharpe(L[s].dropna()) for s in out.index]
out['cat']    = [menu.loc[s, 'Cat.Economic'] for s in out.index]
print(out.sort_values('published_t', ascending=False).round(2).to_string())